# 02_silver_notebook_crm_crb

**Layer**: Silver  
**Source**: `APAC_Reporting_LH.apac_sales_pipeline_fact`  

**Output tables**:
- `APAC_Reporting_LH.CRB_New_Business_Pipeline`
- `APAC_Reporting_LH.CRB_Renewal_Base`
- `APAC_Reporting_LH.APAC_CRB_ALL_RECORDS`
- `APAC_Reporting_LH.CRB_Singapore_Finance`
- `APAC_Reporting_LH.CRB_Hong_Kong_Finance`
- `APAC_Reporting_LH.CRB_India_Finance`
- `APAC_Reporting_LH.CRB_Taiwan_Finance`
- `APAC_Reporting_LH.CRB_Philippines_Finance`
- `APAC_Reporting_LH.CRB_China_Finance`

**Purpose**: Filters and enriches APAC CRM pipeline data for Corporate Risk & Broking (CRB).  
Applies GLOB mapping via SubProduct and Profit Center lookups, FX conversion for assumed pipeline,  
and splits output by opportunity type and country.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, LongType, FloatType, DecimalType, DateType
)
from functools import reduce
import re

In [ ]:
# =============================================================================
# Config — all table names and filter constants in one place
# =============================================================================

# Source tables
SRC_PIPELINE           = "APAC_Reporting_LH.apac_sales_pipeline_fact"
REF_SUBPRODUCT_GLOB    = "APAC_CRM_Analytics_LH.`CRB_Sub Product Class to GLOB Mapping`"
REF_PROFIT_CENTER_GLOB = "APAC_CRM_Analytics_LH.`CRB_Profit Center to GLOB Mapping`"
REF_PIPELINE_PHASE     = "APAC_CRM_Analytics_LH.`CRB_Pipeline Phase Mapping`"
REF_FX_RATES           = "APAC_CRM_Analytics_LH.`CRB_Exchange Rate`"
REF_ASSUMED_PIPELINE   = "APAC_CRM_Analytics_LH.`CRB_Assumed Pipeline`"

# Output tables
TGT_NEW_BUSINESS = "APAC_Reporting_LH.CRB_New_Business_Pipeline"
TGT_RENEWAL_BASE = "APAC_Reporting_LH.CRB_Renewal_Base"
TGT_ALL_RECORDS  = "APAC_Reporting_LH.APAC_CRB_ALL_RECORDS"
TGT_SINGAPORE    = "APAC_Reporting_LH.CRB_Singapore_Finance"
TGT_HONG_KONG    = "APAC_Reporting_LH.CRB_Hong_Kong_Finance"
TGT_INDIA        = "APAC_Reporting_LH.CRB_India_Finance"
TGT_TAIWAN       = "APAC_Reporting_LH.CRB_Taiwan_Finance"
TGT_PHILIPPINES  = "APAC_Reporting_LH.CRB_Philippines_Finance"
TGT_CHINA        = "APAC_Reporting_LH.CRB_China_Finance"

# Japan desk owners — these are reassigned to CRB at filter time
JAPAN_DESK_OWNERS = [
    "Ferrari, Simon", "Fujita, Seiji", "Gonno, Yuta", "Hamashima, Shoichi",
    "Hirai, Tomohiro", "Hirose, Harunobu", "Hoshino, Katsu", "Ikeda, Toshihiko",
    "Inoue, Shuhei", "Kato, Takashi", "Kitashiro, Yasu", "Matsuda, Koji",
    "Miyashita, Hirono", "Morimoto, Hideaki", "Moriya, Kiyoshi", "Moriyama, Kaoru",
    "Nakamura, Hirofumi", "Nakamura, Tatsuro", "Nakano, Mayumi", "Niino, Rie",
    "Sasatani, Jumpei", "Sekiguchi, Taiki", "Shibuya, Shuichi", "Takahashi, Shinto",
    "Tamura, Yoichi", "Yamamoto, Kiyoshi", "Yamamoto, Seishiro",
    "Yasutomi, Toshiyuki", "Yoshihara, Shinji", "Fujimoto, Eiichi", "Komoto, Yusuke",
]
JAPAN_DESK_UPPER = [n.upper() for n in JAPAN_DESK_OWNERS]

# Non-CRB LOBs to exclude (already uppercased for filter)
BUSINESS_LINES_UPPER = [
    "HEALTH AND BENEFITS LOB",
    "HUMAN CAPITAL AND BENEFITS HQ LOB",
    "INTEGRATED AND GLOBAL SOLUTIONS LOB",
    "RETIREMENT LOB",
]

# ProductClass prefixes belonging to non-CRB business lines
PRODUCT_CLASS_PREFIXES = ["H&B", "W&R", "RET:", "INV:", "EX:", "OUT:", "IGS:", "ICT:"]

In [ ]:
def cleanSparkCols(df):
    """Replace spaces, dots, dashes, and parentheses in column names with underscores."""
    newCols = [re.sub(r'[\s\.\-\(\)]', '_', c) for c in df.columns]
    return df.toDF(*newCols)


def cleanseDataframe(df):
    """
    Replicates Alteryx Cleanse tool:
    - Null -> 0 for numeric columns
    - Null -> '' for string columns, then trim, collapse whitespace, uppercase
    """
    stringCols  = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    numericCols = [f.name for f in df.schema.fields
                   if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType, DecimalType))]

    cleansedDf = df.na.fill(0, subset=numericCols)

    for colName in stringCols:
        cleansedDf = cleansedDf.withColumn(
            colName,
            F.upper(
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.coalesce(F.col(colName), F.lit(''))),
                        r'[\t\n\r]', ''
                    ),
                    r'\s+', ' '
                )
            )
        )

    if 'Likelihood_of_Win' in cleansedDf.columns:
        cleansedDf = cleansedDf.withColumn(
            "Likelihood_of_Win",
            F.regexp_replace(F.col("Likelihood_of_Win"), "%", "").cast(DoubleType())
        )

    return cleansedDf


def alignSchema(df, targetColumns):
    """Add missing columns as null and reorder to a consistent target schema."""
    for col in targetColumns:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))
    return df.select(*targetColumns)

In [ ]:
# =============================================================================
# STEP 1: Load source and reference tables
# =============================================================================
print("STEP 1: Loading source tables...")

pipelineDf        = spark.sql(f"SELECT * FROM {SRC_PIPELINE}")
subProductGlobDf  = spark.sql(f"SELECT * FROM {REF_SUBPRODUCT_GLOB}")
globMappingDf     = spark.sql(f"SELECT * FROM {REF_PROFIT_CENTER_GLOB}")
pipelinePhaseDf   = spark.sql(f"SELECT * FROM {REF_PIPELINE_PHASE}")
fxRatesDf         = spark.sql(f"SELECT * FROM {REF_FX_RATES}")
assumedPipelineDf = spark.sql(f"SELECT * FROM {REF_ASSUMED_PIPELINE}")

# Normalise column names — remove spaces, dots, parens
pipelineDf        = cleanSparkCols(pipelineDf)
assumedPipelineDf = cleanSparkCols(assumedPipelineDf)
fxRatesDf         = cleanSparkCols(fxRatesDf)
globMappingDf     = cleanSparkCols(globMappingDf)
subProductGlobDf  = cleanSparkCols(subProductGlobDf)
pipelinePhaseDf   = cleanSparkCols(pipelinePhaseDf)

print(f"Pipeline:         {pipelineDf.count()} rows")
print(f"Assumed pipeline: {assumedPipelineDf.count()} rows")
print(f"FX rates:         {fxRatesDf.count()} rows")
print(f"GLOB mapping:     {globMappingDf.count()} rows")
print(f"SubProduct GLOB:  {subProductGlobDf.count()} rows")
print(f"Pipeline phase:   {pipelinePhaseDf.count()} rows")

pipelineDf.printSchema()
display(pipelineDf.limit(3))

In [ ]:
# =============================================================================
# STEP 2: Filter to CRB records
# =============================================================================
print("STEP 2: Filtering to CRB records...")
initialCount = pipelineDf.count()

# Build product class exclusion filter from config — keeps nulls (pass-through)
productClassFilter = reduce(
    lambda acc, prefix: acc & (
        F.col("ProductClass").isNull() | ~F.upper(F.col("ProductClass")).startswith(prefix)
    ),
    PRODUCT_CLASS_PREFIXES,
    F.lit(True)
)

filteredPipelineDf = pipelineDf.withColumn(
    "Owner_Business_Name",
    F.when(F.upper(F.col("Owner")).isin(JAPAN_DESK_UPPER), "CORPORATE RISK & BROKING")
     .otherwise(F.col("Owner_Business_Name"))
).withColumn(
    "Finance_Level",
    F.when(F.upper(F.col("Owner")).isin(JAPAN_DESK_UPPER), "Japan")
     .otherwise(F.col("Finance_Level"))
).filter(
    (F.col("Data_Version").isNull() | ~F.upper(F.col("Data_Version")).contains("CIS")) &
    (F.col("FormName").isNull() | ~F.upper(F.col("FormName")).contains("ICT")) &
    (F.upper(F.col("Owner_Business_Name")) == "CORPORATE RISK & BROKING") &
    (F.col("FormName").isNull() | (F.upper(F.col("FormName")) != "HWC FORM")) &
    (F.col("Owner_Lob_Name").isNull() | ~F.upper(F.col("Owner_Lob_Name")).isin(BUSINESS_LINES_UPPER)) &
    (F.col("Profit_Center").isNull() | ~F.upper(F.col("Profit_Center")).contains("H&B")) &
    productClassFilter
)

filteredCount = filteredPipelineDf.count()
print(f"Before: {initialCount}  |  After: {filteredCount}  |  Removed: {initialCount - filteredCount}")

filteredPipelineDf.cache()
print("STEP 2 done.")

In [ ]:
# =============================================================================
# STEP 3A: Select, cast, and apply GLOB mapping
# =============================================================================
print("STEP 3A: Enriching pipeline data...")

# Cast before alias — revenue columns coalesced to 0.0, dates to DateType
baseEnrichedDf = filteredPipelineDf.select(
    F.col("Account").cast(StringType()).alias("Account_Name"),
    F.col("Profit_Center").cast(StringType()).alias("Profit_Center"),
    F.col("Project_Start_Date").cast(DateType()).alias("Project_Start_Date"),
    F.col("Service_Office").cast(StringType()).alias("Service_Office"),
    F.col("Service_Office_Country").cast(StringType()).alias("Service_Office_Country"),
    F.col("Service_Region").cast(StringType()).alias("Service_Region"),
    F.col("Source_Campaign").cast(StringType()).alias("Source_Campaign"),
    F.col("TagList").cast(StringType()).alias("TagList"),
    F.col("Tiers").cast(StringType()).alias("Tiers"),
    F.col("Owner").cast(StringType()).alias("Income_Assignee"),
    F.col("Owner_Business_Name").cast(StringType()).alias("Owner_Business_Name"),
    F.col("Owner_Country").cast(StringType()).alias("Owner_Country"),
    F.col("Owner_Lob_Name").cast(StringType()).alias("Owner_Lob_Name"),
    F.col("Owner_Segment_Lob_Name").cast(StringType()).alias("Owner_Segment_Lob_Name"),
    F.col("Colleague_Involved_UPN").cast(StringType()).alias("Owner_Email"),
    F.col("OpptySubType").cast(StringType()).alias("OpptySubType"),
    F.col("ProductSubClass").cast(StringType()).alias("Product_Sub_class"),
    F.col("ProductClass").cast(StringType()).alias("ProductClass"),
    F.col("Pipeline_Phase").cast(StringType()).alias("Pipeline_Phase"),
    F.col("CCY").cast(StringType()).alias("CCY"),
    F.col("CloseDate").cast(DateType()).alias("Close_Date"),
    F.col("Account_Country").cast(StringType()).alias("Account_Country"),
    F.col("CreatedOn").cast(DateType()).alias("CreatedOn"),
    F.col("Data_Version").cast(StringType()).alias("Data_Version"),
    F.col("Description").cast(StringType()).alias("OpptySummary"),
    F.coalesce(F.col("Est__Revenue__LCY_"), F.lit(0.0)).cast(DoubleType()).alias("Local_Est_Revenue"),
    F.coalesce(F.col("Est__Revenue__USD_"), F.lit(0.0)).cast(DoubleType()).alias("Total_Est_Revenue"),
    F.coalesce(F.col("Wt__Revenue__LCY_"), F.lit(0.0)).cast(DoubleType()).alias("Local_Wt_Revenue"),
    F.coalesce(F.col("Wt__Revenue__USD_"), F.lit(0.0)).cast(DoubleType()).alias("Total_Wt_Revenue"),
    F.col("Finance_Level").cast(StringType()).alias("Country"),
    F.col("First_Income_Date").cast(DateType()).alias("First_Income_Date"),
    F.col("FormName").cast(StringType()).alias("FormName"),
    F.col("Frequency").cast(StringType()).alias("Frequency"),
    F.col("GCID").cast(StringType()).alias("GCID"),
    F.col("GUID").cast(StringType()).alias("GUID"),
    F.col("Industry").cast(StringType()).alias("Industry"),
    F.coalesce(
        F.regexp_replace(F.col("Likelihood_of_Win"), "%", "").cast(DoubleType()),
        F.lit(0.0)
    ).alias("Probability"),
    F.col("ModifiedOn").cast(DateType()).alias("Modified_On"),
    F.col("Opportunity_Owner").cast(StringType()).alias("Opportunity_Owner"),
    F.col("OpptyID").cast(StringType()).alias("OpportunityID"),
    F.col("OpptyState").cast(StringType()).alias("Status_Reason"),
    F.col("OpptyStatus").cast(StringType()).alias("OpptyStatus"),
    F.col("OpptySummary").cast(StringType()).alias("Opportunity_Name"),
    F.col("OpptyType").cast(StringType()).alias("Opportunity_Type"),
    F.col("Primary_SIC_Code").cast(StringType()).alias("Sub_Industry"),
).withColumn(
    "Opp_Link",
    F.concat(
        F.lit("https://wtwcrb.crm.dynamics.com/main.aspx?appid=af96b65c-0084-ea11-a813-000d3a579b83&forceUCI=1&pagetype=entityrecord&etn=opportunity&id="),
        F.col("GUID")
    )
)

# SubProduct -> GLOB mapping (TRIM+UPPER on both sides)
enrichedWithSubproductDf = baseEnrichedDf.join(
    subProductGlobDf,
    F.trim(F.upper(baseEnrichedDf["Product_Sub_class"])) == F.trim(F.upper(subProductGlobDf["D365_Product_Sub_Class"])),
    "left"
).withColumnRenamed("GLOB_Mapping", "GLOBs").drop("D365_Product_Sub_Class")

# Profit Center -> GLOB mapping (TRIM+UPPER on both sides)
enrichedWithPcDf = enrichedWithSubproductDf.join(
    globMappingDf,
    F.trim(F.upper(enrichedWithSubproductDf["Profit_Center"])) == F.trim(F.upper(globMappingDf["D365_Profit_Center"])),
    "left"
).withColumnRenamed("Profit_Center_to_GLOB_Mapping", "PC_GLOBs").drop("D365_Profit_Center")

# Use SubProduct GLOB first; fall back to Profit Center GLOB if null or unclassified
# Remap Cyber -> FINEX per business rule
enrichedDf = enrichedWithPcDf.withColumn(
    "GLOBs",
    F.when(F.col("GLOBs").isNull() | (F.upper(F.col("GLOBs")) == "UNCLASSIFIED"), F.col("PC_GLOBs"))
     .otherwise(F.col("GLOBs"))
).withColumn(
    "GLOBs",
    F.when(F.upper(F.col("GLOBs")) == "CYBER", "FINEX").otherwise(F.col("GLOBs"))
).drop("PC_GLOBs")

enrichedCount = enrichedDf.count()
print(f"Enriched records: {enrichedCount}")
enrichedDf.cache()
print("STEP 3A done.")

In [ ]:
# =============================================================================
# STEP 3B: Capture unmapped keys and register back to reference tables
# Mirrors the Alteryx append-back pattern — new keys are inserted with a blank
# mapping value so they can be filled in manually via the Lakehouse table editor.
# Note: MERGE uses the original spaced column names from the Delta table.
# =============================================================================
print("STEP 3B: Checking for unmapped reference keys...")

# SubProduct GLOB — rows where join returned no match, exclude nulls and blanks
unmappedSubProduct = enrichedWithSubproductDf.filter(
    F.col("GLOBs").isNull() &
    F.col("Product_Sub_class").isNotNull() &
    (F.trim(F.col("Product_Sub_class")) != "")
).select(
    F.trim(F.upper(F.col("Product_Sub_class"))).alias("D365_Product_Sub_Class")
).distinct()

unmappedSpCount = unmappedSubProduct.count()
print(f"Unmapped SubProduct keys: {unmappedSpCount}")

if unmappedSpCount > 0:
    display(unmappedSubProduct)
    unmappedSubProduct.createOrReplaceTempView("_tmp_unmapped_subproduct")
    spark.sql("""
        MERGE INTO APAC_CRM_Analytics_LH.`CRB_Sub Product Class to GLOB Mapping` AS t
        USING _tmp_unmapped_subproduct AS s
        ON UPPER(TRIM(t.`D365 Product Sub Class`)) = UPPER(TRIM(s.D365_Product_Sub_Class))
        WHEN NOT MATCHED THEN INSERT (`D365 Product Sub Class`, `GLOB Mapping`) VALUES (s.D365_Product_Sub_Class, '')
    """)
    print(f"Inserted {unmappedSpCount} new key(s) into SubProduct GLOB mapping with blank value.")
    print("Action required: open APAC_CRM_Analytics_LH > CRB_Sub Product Class to GLOB Mapping and fill blank rows.")

# Profit Center GLOB — rows where join returned no match, exclude nulls and blanks
unmappedProfitCenter = enrichedWithPcDf.filter(
    F.col("PC_GLOBs").isNull() &
    F.col("Profit_Center").isNotNull() &
    (F.trim(F.col("Profit_Center")) != "")
).select(
    F.trim(F.upper(F.col("Profit_Center"))).alias("D365_Profit_Center")
).distinct()

unmappedPcCount = unmappedProfitCenter.count()
print(f"Unmapped Profit Center keys: {unmappedPcCount}")

if unmappedPcCount > 0:
    display(unmappedProfitCenter)
    unmappedProfitCenter.createOrReplaceTempView("_tmp_unmapped_profit_center")
    spark.sql("""
        MERGE INTO APAC_CRM_Analytics_LH.`CRB_Profit Center to GLOB Mapping` AS t
        USING _tmp_unmapped_profit_center AS s
        ON UPPER(TRIM(t.`D365 Profit Center`)) = UPPER(TRIM(s.D365_Profit_Center))
        WHEN NOT MATCHED THEN INSERT (`D365 Profit Center`, `Profit Center to GLOB Mapping`) VALUES (s.D365_Profit_Center, '')
    """)
    print(f"Inserted {unmappedPcCount} new key(s) into Profit Center GLOB mapping with blank value.")
    print("Action required: open APAC_CRM_Analytics_LH > CRB_Profit Center to GLOB Mapping and fill blank rows.")

print("STEP 3B done.")

In [ ]:
# =============================================================================
# STEP 3C: Process assumed pipeline and union with main pipeline
# =============================================================================
print("STEP 3C: Processing assumed pipeline...")
assumedPreCount = assumedPipelineDf.count()

# FX conversion — TRIM+UPPER on both sides
assumedWithFxDf = assumedPipelineDf.join(
    fxRatesDf,
    F.trim(F.upper(assumedPipelineDf["Currency"])) == F.trim(F.upper(fxRatesDf["Currency"])),
    "left"
)

# Convert assumed amounts to USD and align to main pipeline schema
# Use a temp column to avoid referencing the result column multiple times in select
assumedTransformedDf = assumedWithFxDf.withColumn(
    "_totalEstRev",
    F.coalesce(F.col("Assumed_New_Business_Pipeline"), F.lit(0.0)) *
    F.coalesce(F.col("Avg_Rate"), F.lit(1.0))
).select(
    F.concat(F.col("Country"), F.lit(" "), F.col("GLOB"), F.lit(" Assumed Pipeline")).cast(StringType()).alias("Account_Name"),
    F.lit("Revenue Plug").cast(StringType()).alias("Product_Sub_class"),
    F.lit("Revenue Plug").cast(StringType()).alias("Opportunity_Name"),
    F.lit("Open").cast(StringType()).alias("Status_Reason"),
    F.col("_totalEstRev").cast(DoubleType()).alias("Total_Est_Revenue"),
    F.col("_totalEstRev").cast(DoubleType()).alias("Total_Wt_Revenue"),
    F.col("_totalEstRev").cast(DoubleType()).alias("Local_Est_Revenue"),
    F.col("_totalEstRev").cast(DoubleType()).alias("Local_Wt_Revenue"),
    F.col("GLOB").cast(StringType()).alias("GLOBs"),
    F.col("Pipeline_Period").cast(DateType()).alias("First_Income_Date"),
    F.col("Country").cast(StringType()).alias("Country"),
    F.current_date().alias("CreatedOn"),
)

# Preserve main pipeline column order; append any extra assumed-only columns after
allColumns       = list(dict.fromkeys(enrichedDf.columns + assumedTransformedDf.columns))
mainAlignedDf    = alignSchema(enrichedDf, allColumns)
assumedAlignedDf = alignSchema(assumedTransformedDf, allColumns)

combinedDf = mainAlignedDf.union(assumedAlignedDf).withColumn(
    "Pipeline_Category",
    F.when(F.upper(F.col("Account_Name")).contains("ASSUMED PIPELINE"), "Assumed Pipeline")
     .otherwise("D365 Pipeline")
)

combinedCount = combinedDf.count()
expectedTotal = enrichedCount + assumedPreCount
print(f"Main: {enrichedCount}  |  Assumed: {assumedPreCount}  |  Combined: {combinedCount}")

if combinedCount != expectedTotal:
    print(f"WARNING: Expected {expectedTotal}, got {combinedCount} — check union logic.")

combinedDf.cache()
print("STEP 3C done.")

In [ ]:
# =============================================================================
# STEP 4: Split into output datasets
# =============================================================================
print("STEP 4: Splitting into output datasets...")

newBusinessPipelineDf = combinedDf.filter(
    F.col("Opportunity_Type").isNotNull() &
    (F.upper(F.col("Opportunity_Type")) != "RENEWAL")
)
renewalPipelineDf = combinedDf.filter(
    F.upper(F.col("Opportunity_Type")) == "RENEWAL"
)
singaporeDf   = combinedDf.filter(F.upper(F.col("Country")) == "SINGAPORE")
hongKongDf    = combinedDf.filter(F.upper(F.col("Country")) == "HONG KONG")
indiaDf       = combinedDf.filter(F.upper(F.col("Service_Office_Country")) == "INDIA")
taiwanDf      = combinedDf.filter(F.upper(F.col("Country")) == "TAIWAN")
philippinesDf = combinedDf.filter(F.upper(F.col("Country")) == "PHILIPPINES")
chinaDf       = combinedDf.filter(F.upper(F.col("Country")) == "CHINA")

# Compute counts before write to avoid recomputation
nbCount  = newBusinessPipelineDf.count()
rnCount  = renewalPipelineDf.count()
sgCount  = singaporeDf.count()
hkCount  = hongKongDf.count()
inCount  = indiaDf.count()
twCount  = taiwanDf.count()
phCount  = philippinesDf.count()
cnCount  = chinaDf.count()

print(f"New Business:  {nbCount}")
print(f"Renewal:       {rnCount}")
print(f"Singapore:     {sgCount}")
print(f"Hong Kong:     {hkCount}")
print(f"India:         {inCount}")
print(f"Taiwan:        {twCount}")
print(f"Philippines:   {phCount}")
print(f"China:         {cnCount}")
print("STEP 4 done.")

In [ ]:
# =============================================================================
# STEP 5: Write to Silver Lakehouse
# =============================================================================
print("STEP 5: Writing to Silver Lakehouse...")

newBusinessPipelineDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_NEW_BUSINESS)
print(f"Saved {TGT_NEW_BUSINESS}: {nbCount} rows")

renewalPipelineDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_RENEWAL_BASE)
print(f"Saved {TGT_RENEWAL_BASE}: {rnCount} rows")

combinedDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_ALL_RECORDS)
print(f"Saved {TGT_ALL_RECORDS}: {combinedCount} rows")

singaporeDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_SINGAPORE)
print(f"Saved {TGT_SINGAPORE}: {sgCount} rows")

hongKongDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_HONG_KONG)
print(f"Saved {TGT_HONG_KONG}: {hkCount} rows")

indiaDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_INDIA)
print(f"Saved {TGT_INDIA}: {inCount} rows")

taiwanDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_TAIWAN)
print(f"Saved {TGT_TAIWAN}: {twCount} rows")

philippinesDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_PHILIPPINES)
print(f"Saved {TGT_PHILIPPINES}: {phCount} rows")

chinaDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT_CHINA)
print(f"Saved {TGT_CHINA}: {cnCount} rows")

print("STEP 5 complete.")